In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Konfiguracja zarządzania szumem w Estimator

<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany przy użyciu następujących wymagań.
Zalecamy używanie tych lub nowszych wersji.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Istnieje kilka sposobów zarządzania szumem, zazwyczaj za pomocą różnych technik łagodzenia błędów i tłumienia błędów, aby zapobiec wystąpieniu błędów. Techniki te zwykle powodują narzut w fazie wstępnego przetwarzania. Dlatego ważne jest, aby znaleźć równowagę między doskonaleniem wyników a zapewnieniem, że zadanie zostanie ukończone w rozsądnym czasie.

Estimator obsługuje następujące techniki zarządzania szumem. Zobacz [Techniki łagodzenia i tłumienia błędów](error-mitigation-and-suppression-techniques), aby zapoznać się z wyjaśnieniem każdej z nich. Zobacz sekcję [Niestandardowe ustawienia błędów](#advanced-error), aby uzyskać instrukcje włączania tych technik.

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Poziom odporności
`resilience_level` określa, jak duża odporność na błędy ma zostać zbudowana. Wyższe poziomy generują dokładniejsze wyniki kosztem dłuższego czasu przetwarzania. Poziomy odporności można wykorzystać do skonfigurowania kompromisu kosztów/dokładności przy stosowaniu zarządzania szumem do zapytania prymitywu. Zarządzanie szumem redukuje błędy (obciążenie) w wynikach przez przetwarzanie wyjść z kolekcji, czyli zespołu powiązanych obwodów. Stopień redukcji błędów zależy od zastosowanej metody. Poziom odporności abstrahuje szczegółowy wybór metody zarządzania szumem, aby umożliwić użytkownikom rozważanie kompromisu kosztów/dokładności odpowiedniego dla ich aplikacji.

Mając to na uwadze, każdy poziom odpowiada metodzie lub metodom o rosnącym poziomie narzutu pobierania próbek kwantowych, co pozwala ci eksperymentować z różnymi kompromisami czasu i dokładności. Poniższa tabela pokazuje, które poziomy i odpowiadające im metody są dostępne dla każdego z prymitywów.

<span id="resilience-table"></span>

| Poziom odporności | Opis                                                                                                   | Technika                                                                  |
|-------------------|--------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                 | Brak łagodzenia                                                                                        | Brak                                                                      |
| 1 [Domyślny]      | Minimalne koszty łagodzenia: łagodzenie błędów odczytu                                                 | [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) z twirlingiem pomiarowym           |
| 2                 | Średnie koszty łagodzenia. Zazwyczaj redukuje obciążenie estymatorów, ale nie gwarantuje zerowego obciążenia. | Poziom 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) i twirling bramek

> **Info:** Poziomy odporności są obecnie w fazie beta, więc narzut pobierania próbek i
> jakość rozwiązań będą się różnić w zależności od obwodu. Nowe funkcje,
> zaawansowane opcje i narzędzia zarządzania będą wydawane na bieżąco.
> Nie ma gwarancji, że określone metody zarządzania szumem
> będą stosowane na każdym poziomie odporności.

### Konfiguracja Estimator z poziomami odporności
Możesz używać poziomów odporności do określania technik zarządzania szumem lub możesz indywidualnie ustawiać niestandardowe techniki, jak opisano w [Niestandardowe ustawienia błędów](#advanced-error).

> **Note:** Wszelkie opcje, które ustawiasz ręcznie, oprócz poziomu odporności, są stosowane jako uzupełnienie podstawowego zestawu opcji zdefiniowanych przez poziom odporności. Dlatego w zasadzie możesz ustawić poziom odporności na 1, ale następnie wyłączyć łagodzenie błędów pomiarowych, choć nie jest to zalecane.
> 
> Na przykład ustawienie poziomu odporności na 0 wyłącza `zne_mitigation`, ale `estimator.options.resilience.zne_mitigation = True` nadpisuje tę wartość.

### Przykład
Poniższy kod włącza ZNE, TREX i twirling bramek przez
ustawienie `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Niestandardowe ustawienia zarządzania szumem
Możesz włączać i wyłączać poszczególne metody zarządzania szumem, korzystając z [opcji Estimator](/guides/estimator-options).

> **Note:** Nie wszystkie opcje działają razem ze wszystkimi typami obwodów. Zobacz [tabelę kompatybilności funkcji](estimator-options#options-compatibility-table), aby uzyskać szczegółowe informacje.

### Przykład

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>